In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

In [ ]:
df = pd.read_csv("./data/IMDB Dataset.csv")

In [ ]:
df

In [ ]:
df.sample(5)

In [ ]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)

In [ ]:
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [ ]:
train_y.value_counts()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

train_x_vector = tfidf.fit_transform(train_x)

In [ ]:
train_x.shape

In [ ]:
train_x_vector.shape

In [ ]:
type(train_x_vector)

In [ ]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]

In [ ]:
primera_resenia

In [ ]:
train_x.iloc[0]

In [ ]:
primera_resenia[primera_resenia != 0]

In [ ]:
train_x.iloc[0]

In [ ]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

In [ ]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))

In [ ]:
print(svc.score(test_x_vector, test_y))

In [ ]:
from sklearn.metrics import f1_score

f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

In [ ]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

## Comparación de cuatro clasificadores

Esta ampliación conserva las celdas de la clase y compara SVM, regresión logística, GaussianNB y árbol de decisión. Se utilizan exactamente el mismo entrenamiento, test y representación TF-IDF del cuaderno original.

GaussianNB requiere datos densos. Para no convertir la matriz completa a un arreglo enorme, se entrena incrementalmente con partial_fit y se predice por lotes de 64 reseñas. Es una adaptación de memoria; el suavizado de varianza puede diferir ligeramente del ajuste en un único lote.

Accuracy significa proporción total de aciertos, mientras que precision mide la proporción de aciertos dentro de las predicciones de una clase. No son la misma métrica.

Esta comparación es exploratoria: el test de la clase ya fue observado. Para demostrar una ventaja generalizable, habría que seleccionar y ajustar modelos mediante validación o validación cruzada dentro del entrenamiento y evaluar una sola vez en un test independiente.

In [ ]:
from pathlib import Path
from time import perf_counter
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SALIDA = Path('resultados_pln')
SALIDA.mkdir(exist_ok=True)
clases = np.array(['negative', 'positive'])
test_x_vector = tfidf.transform(test_x)
assert train_x_vector.shape[1] == test_x_vector.shape[1]
assert train_x_vector.shape[0] == len(train_y)
assert test_x_vector.shape[0] == len(test_y)
print('Entrenamiento:', train_x_vector.shape, '| Prueba:', test_x_vector.shape)

### Funciones para GaussianNB

toarray() se aplica únicamente a cada lote, no a todo el vocabulario para todas las reseñas a la vez. Los otros tres modelos trabajan directamente con la matriz dispersa.

In [ ]:
def ajustar_gaussian_por_lotes(modelo, X, y, lote=64):
    y_array = np.asarray(y)
    for inicio in range(0, X.shape[0], lote):
        fin = min(inicio + lote, X.shape[0])
        modelo.partial_fit(X[inicio:fin].toarray(), y_array[inicio:fin], classes=clases)
    return modelo

def predecir_gaussian_por_lotes(modelo, X, lote=64):
    return np.concatenate([
        modelo.predict(X[inicio:inicio + lote].toarray())
        for inicio in range(0, X.shape[0], lote)
    ])

In [ ]:
modelos = {
    'SVM lineal': SVC(kernel='linear'),
    'Regresión logística': LogisticRegression(max_iter=2000, random_state=42),
    'GaussianNB': GaussianNB(),
    'Árbol de decisión': DecisionTreeClassifier(random_state=42),
}
resultados = []
predicciones = {}
for nombre, modelo in modelos.items():
    inicio = perf_counter()
    if nombre == 'GaussianNB':
        ajustar_gaussian_por_lotes(modelo, train_x_vector, train_y)
    else:
        modelo.fit(train_x_vector, train_y)
    tiempo_entrenamiento = perf_counter() - inicio
    inicio = perf_counter()
    pred = (predecir_gaussian_por_lotes(modelo, test_x_vector)
            if nombre == 'GaussianNB' else modelo.predict(test_x_vector))
    tiempo_prediccion = perf_counter() - inicio
    predicciones[nombre] = pred
    assert len(pred) == len(test_y)
    assert set(pred).issubset(set(clases))
    resultados.append({
        'Modelo': nombre,
        'Accuracy': accuracy_score(test_y, pred),
        'Precision macro': precision_score(test_y, pred, average='macro', zero_division=0),
        'Recall macro': recall_score(test_y, pred, average='macro', zero_division=0),
        'F1 macro': f1_score(test_y, pred, average='macro', zero_division=0),
        'Entrenamiento (s)': tiempo_entrenamiento,
        'Predicción (s)': tiempo_prediccion,
    })
    print(nombre)
    print(classification_report(test_y, pred, labels=clases, zero_division=0))
    reporte = pd.DataFrame(classification_report(test_y, pred, labels=clases, output_dict=True, zero_division=0)).T
    reporte.to_csv(SALIDA / (nombre.replace(' ', '_') + '_reporte.csv'))

comparacion = pd.DataFrame(resultados).sort_values(['F1 macro', 'Accuracy'], ascending=False)
display(comparacion.round(4))
comparacion.to_csv(SALIDA / 'comparacion_modelos.csv', index=False)

### Visualizar los resultados

Se muestran las mismas métricas para todos los modelos. En las matrices de confusión las filas representan las clases reales y las columnas las predichas.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
comparacion.set_index('Modelo')[['Accuracy', 'F1 macro']].plot.bar(ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel('Puntuación')
ax.set_title('Comparación exploratoria en el mismo conjunto de prueba')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(SALIDA / 'comparacion_modelos.png', dpi=180)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, (nombre, pred) in zip(axes.flat, predicciones.items()):
    matriz = confusion_matrix(test_y, pred, labels=clases)
    ConfusionMatrixDisplay(matriz, display_labels=clases).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nombre)
plt.tight_layout()
plt.savefig(SALIDA / 'matrices_confusion.png', dpi=180)
plt.show()

### Probar reseñas nuevas

Se usa el mismo vectorizador del entrenamiento. Estas frases son demostraciones; tres ejemplos no sustituyen la evaluación cuantitativa.

In [ ]:
ejemplos = pd.Series([
    'A good movie',
    'An excellent movie',
    'I did not like this movie at all',
], name='Reseña')
vectores_ejemplo = tfidf.transform(ejemplos)
tabla_ejemplos = ejemplos.to_frame()
for nombre, modelo in modelos.items():
    tabla_ejemplos[nombre] = (predecir_gaussian_por_lotes(modelo, vectores_ejemplo)
                             if nombre == 'GaussianNB' else modelo.predict(vectores_ejemplo))
display(tabla_ejemplos)

In [ ]:
mejor_observado = comparacion.iloc[0]
svm_observado = comparacion.loc[comparacion['Modelo'] == 'SVM lineal'].iloc[0]
diferencia = mejor_observado['F1 macro'] - svm_observado['F1 macro']
print('Mayor F1 macro observado:', mejor_observado['Modelo'])
print('F1 macro:', round(mejor_observado['F1 macro'], 4))
print('Diferencia respecto a SVM:', round(diferencia, 4))
print('Resultado exploratorio para esta muestra y configuración; no demuestra superioridad general.')

## Conclusión y entrega

Después de ejecutar todas las celdas, escribe una conclusión con la tabla real: ¿SVM obtuvo el mayor F1 macro? ¿Qué modelo fue más rápido? ¿Qué errores aparecen en cada clase? No se afirma un ganador antes de ejecutar el cuaderno ni se compara directamente con el 87.06% de las notas, porque el vectorizador del notebook original puede diferir del de la explicación.

Guarda el notebook con sus salidas, súbelo al repositorio y toma una captura que muestre el archivo actualizado y el commit. Esa captura se entrega en la plataforma del curso. El CSV original no se modifica.